# 任务五：基于双层LSTM的Seq2Seq德英机器翻译
本任务实现了一个基于**双层LSTM**的编码器-解码器（Encoder-Decoder）架构，完成德语到英语的端到端机器翻译任务。

## 1. 环境配置与依赖导入
导入PyTorch深度学习框架及相关工具库，自动配置运行设备（优先使用GPU，无GPU则使用CPU）。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import spacy
import numpy as np
import random
import math
import time

# 设置随机种子，保证结果可复现
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 选择设备（有GPU用GPU，没有就用CPU）
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用设备:", device)

使用设备: cpu


In [3]:
# 加载英语和德语分词模型
nlp_en = spacy.load('en_core_web_sm')
nlp_de = spacy.load('de_core_news_sm')

# 定义德语分词函数（源语言）
def tokenize_de(text):
    return [tok.text for tok in nlp_de.tokenizer(text)]

# 定义英语分词函数（目标语言）
def tokenize_en(text):
    return [tok.text for tok in nlp_en.tokenizer(text)]

print("✅ 分词模型加载成功！")

✅ 分词模型加载成功！


## 2. 数据集准备
使用德英双语平行语料作为训练集，包含简单的日常句子对，用于训练翻译模型。

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import spacy

train_data = [
    ("Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.", "Two young White males are outside near many bushes."),
    ("Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.", "Several men in hard hats are operating a giant pulley system."),
    ("Ein kleines Mädchen klettert in ein Spielhaus aus Holz.", "A little girl climbing into a wooden playhouse."),
    ("Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.", "A man in a blue shirt is standing on a ladder cleaning a window."),
    ("Zwei Männer stehen am Herd und bereiten Essen zu.", "Two men are at the stove preparing food."),
    ("Ein Mann in grün hält eine Gitarre und spielt.", "A man in green holds a guitar while the other man sings."),
    ("Ein Mann lächelt einen ausgestopften Löwen an.", "A man is smiling at a stuffed lion."),
    ("Ein schickes Mädchen spricht mit dem Handy während sie läuft.", "A trendy girl talking on her cellphone while gliding slowly down the street."),
    ("Eine Frau mit einer großen Geldbörse geht an einem Tor vorbei.", "A woman with a large purse is walking by a gate."),
    ("Jungen tanzen mitten in der Nacht auf Pfosten.", "Boys dancing on poles in the middle of the night."),
    ("Fünf Leute in Winterjacken und mit Helmen stehen im Schnee mit Schneemobilen.", "Five people wearing winter jackets and helmets stand in the snow with snowmobiles."),
    ("Leute reparieren das Dach eines Hauses.", "People are fixing the roof of a house."),
    ("Ein Mann lächelt und hält eine gestreifte Katze.", "A man smiling and holding a tabby cat."),
    ("Ein Mann in einer schwarzen Jacke und einer dunklen Hose geht an einem Tor vorbei.", "A man in a black jacket and dark pants is walking past a gate."),
    ("Ein Mann mit einem blauen Hut steht vor einem Graffiti.", "A man with a blue hat is standing in front of a graffiti."),
    ("Ein Junge in einem roten Shirt schlägt einen Baseball.", "A boy in a red shirt is hitting a baseball."),
    ("Ein Mann in einem schwarzen Anzug hält eine Kamera.", "A man in a black suit is holding a camera."),
    ("Ein Hund läuft durch ein Feld mit gelben Blumen.", "A dog is running through a field of yellow flowers."),
    ("Eine Gruppe von Menschen steht vor einem Gebäude.", "A group of people are standing in front of a building."),
    ("Ein Junge fährt mit seinem Fahrrad auf einer Straße.", "A boy is riding his bicycle on a street."),
    ("Eine Frau backt Kuchen in der Küche.", "A woman is baking a cake in the kitchen."),
    ("Ein alter Mann sitzt auf einer Bank und liest eine Zeitung.", "An old man is sitting on a bench reading a newspaper."),
    ("Zwei Kinder spielen mit einem Ball im Garten.", "Two children are playing with a ball in the garden."),
    ("Ein Vogel fliegt über den blauen Himmel.", "A bird is flying over the blue sky."),
    ("Ein Auto fährt auf einer Autobahn.", "A car is driving on a highway."),
    ("Ein Mann trinkt Kaffee am Tisch.", "A man is drinking coffee at the table."),
    ("Eine Frau trägt einen roten Mantel.", "A woman is wearing a red coat."),
    ("Eine Katze schläft auf dem Sofa.", "A cat is sleeping on the sofa."),
    ("Ein Fisch schwimmt im Aquarium.", "A fish is swimming in the aquarium."),
    ("Ein Baum steht im Park.", "A tree is standing in the park."),
    ("Ein Buch liegt auf dem Tisch.", "A book is lying on the table."),
    ("Ein Stuhl steht vor dem Tisch.", "A chair is standing in front of the table."),
    ("Ein Fenster ist offen.", "A window is open."),
    ("Eine Tür ist geschlossen.", "A door is closed."),
    ("Die Sonne scheint.", "The sun is shining."),
    ("Es regnet.", "It is raining."),
    ("Es schneit.", "It is snowing."),
    ("Der Wind weht.", "The wind is blowing."),
    ("Ein Mann geht zur Arbeit.", "A man is going to work."),
    ("Eine Frau kommt nach Hause.", "A woman is coming home."),
    ("Ein Junge geht zur Schule.", "A boy is going to school."),
    ("Ein Mädchen geht zum Kindergarten.", "A girl is going to kindergarten."),
    ("Ein Lehrer unterrichtet im Klassenzimmer.", "A teacher is teaching in the classroom."),
    ("Ein Schüler lernt für eine Prüfung.", "A student is studying for an exam."),
    ("Ein Arzt untersucht einen Patienten.", "A doctor is examining a patient."),
    ("Ein Polizist steht an der Kreuzung.", "A policeman is standing at the intersection."),
    ("Ein Feuerwehrmann löscht ein Feuer.", "A firefighter is putting out a fire."),
    ("Ein Koch kocht in einem Restaurant.", "A chef is cooking in a restaurant."),
    ("Ein Kellner bringt Essen zum Tisch.", "A waiter is bringing food to the table."),
    ("Ein Fahrer fährt einen Bus.", "A driver is driving a bus."),
    ("Ein Pilot fliegt ein Flugzeug.", "A pilot is flying an airplane."),
    ("Ein Astronaut fliegt im Weltraum.", "An astronaut is flying in space."),
    ("Ein Künstler malt ein Bild.", "An artist is painting a picture."),
    ("Ein Musiker spielt ein Klavier.", "A musician is playing a piano."),
    ("Ein Schriftsteller schreibt ein Buch.", "A writer is writing a book."),
    ("Ein Schauspieler spielt in einem Film.", "An actor is acting in a movie."),
    ("Ein Sportler läuft einen Marathon.", "An athlete is running a marathon."),
    ("Ein Fußballer spielt Fußball.", "A football player is playing football."),
    ("Ein Basketballer spielt Basketball.", "A basketball player is playing basketball."),
    ("Ein Tennisspieler spielt Tennis.", "A tennis player is playing tennis."),
    ("Ein Schwimmer schwimmt im Pool.", "A swimmer is swimming in the pool."),
    ("Ein Skifahrer fährt Ski.", "A skier is skiing."),
    ("Ein Snowboarder fährt Snowboard.", "A snowboarder is snowboarding."),
    ("Ein Surfer surft im Meer.", "A surfer is surfing in the sea."),
    ("Ein Taucher taucht unter Wasser.", "A diver is diving underwater."),
    ("Ein Bergsteiger klettert einen Berg.", "A mountaineer is climbing a mountain."),
    ("Ein Wanderer wandert im Wald.", "A hiker is hiking in the forest."),
    ("Ein Camper zeltet am See.", "A camper is camping by the lake."),
    ("Ein Angler angelt am Fluss.", "A fisherman is fishing by the river."),
    ("Ein Bauer arbeitet auf dem Feld.", "A farmer is working in the field."),
    ("Ein Gärtner gießt die Blumen.", "A gardener is watering the flowers."),
    ("Ein Mechaniker repariert ein Auto.", "A mechanic is repairing a car."),
    ("Ein Elektriker repariert eine Lampe.", "An electrician is repairing a lamp."),
    ("Ein Klempner repariert ein Rohr.", "A plumber is repairing a pipe."),
    ("Ein Tischler macht einen Tisch.", "A carpenter is making a table."),
    ("Ein Maurer baut eine Wand.", "A mason is building a wall."),
    ("Ein Maler streicht eine Wand.", "A painter is painting a wall."),
    ("Ein Architekt entwirft ein Gebäude.", "An architect is designing a building."),
    ("Ein Ingenieur baut eine Brücke.", "An engineer is building a bridge."),
    ("Ein Wissenschaftler forscht im Labor.", "A scientist is researching in the laboratory."),
    ("Ein Informatiker programmiert einen Computer.", "A computer scientist is programming a computer."),
    ("Ein Mann läuft schnell.", "A man is running fast."),
    ("Ein Mann läuft langsam.", "A man is running slowly."),
    ("Eine Frau spricht laut.", "A woman is speaking loudly."),
    ("Eine Frau spricht leise.", "A woman is speaking quietly."),
    ("Ein Kind lacht laut.", "A child is laughing loudly."),
    ("Ein Kind weint leise.", "A child is crying quietly."),
    ("Ein Hund bellt laut.", "A dog is barking loudly."),
    ("Eine Katze miaut leise.", "A cat is meowing quietly."),
    ("Ein Vogel singt schön.", "A bird is singing beautifully."),
    ("Ein Auto fährt schnell.", "A car is driving fast."),
    ("Ein Auto fährt langsam.", "A car is driving slowly."),
    ("Ein Zug fährt sehr schnell.", "A train is driving very fast."),
    ("Ein Flugzeug fliegt sehr hoch.", "An airplane is flying very high."),
    ("Ein Schiff fährt langsam.", "A ship is sailing slowly."),
    ("Ein Buch ist interessant.", "A book is interesting."),
    ("Ein Buch ist langweilig.", "A book is boring."),
    ("Ein Film ist spannend.", "A movie is exciting."),
    ("Ein Film ist traurig.", "A movie is sad."),
    ("Ein Lied ist schön.", "A song is beautiful."),
    ("Das Essen ist lecker.", "The food is delicious."),
    ("Das Wetter ist schön.", "The weather is nice."),
    ("Die Landschaft ist schön.", "The landscape is beautiful."),
    ("Die Stadt ist laut.", "The city is noisy."),
    ("Das Land ist ruhig.", "The countryside is quiet."),
    ("Ich bin glücklich.", "I am happy."),
    ("Ich bin traurig.", "I am sad."),
    ("Ich bin müde.", "I am tired."),
    ("Ich bin hungrig.", "I am hungry."),
    ("Ich bin durstig.", "I am thirsty."),
    ("Ich bin gesund.", "I am healthy."),
    ("Ich bin beschäftigt.", "I am busy."),
    ("Ich bin frei.", "I am free."),
    ("Ich bin allein.", "I am alone."),
    ("Ich bin mit Freunden zusammen.", "I am with friends."),
    ("Ich bin zu Hause.", "I am at home."),
    ("Ich bin bei der Arbeit.", "I am at work."),
    ("Ich bin in der Schule.", "I am at school."),
    ("Ich bin im Park.", "I am in the park."),
    ("Ich bin im Supermarkt.", "I am in the supermarket."),
    ("Ich bin im Restaurant.", "I am in the restaurant."),
    ("Ich bin im Kino.", "I am at the cinema."),
    ("Ich bin im Theater.", "I am at the theater."),
    ("Ich bin im Museum.", "I am in the museum."),
    ("Ich bin im Krankenhaus.", "I am in the hospital."),
    ("Ich bin im Büro.", "I am in the office."),
    ("Ich bin im Auto.", "I am in the car."),
    ("Ich bin im Zug.", "I am on the train."),
    ("Ich bin im Flugzeug.", "I am on the airplane."),
    ("Ich gehe nach Hause.", "I am going home."),
    ("Ich gehe zur Arbeit.", "I am going to work."),
    ("Ich gehe zur Schule.", "I am going to school."),
    ("Ich gehe zum Park.", "I am going to the park."),
    ("Ich gehe zum Supermarkt.", "I am going to the supermarket."),
    ("Ich gehe zum Restaurant.", "I am going to the restaurant."),
    ("Ich gehe zum Kino.", "I am going to the cinema."),
    ("Ich gehe zum Theater.", "I am going to the theater."),
    ("Ich gehe zum Museum.", "I am going to the museum."),
    ("Ich gehe zum Krankenhaus.", "I am going to the hospital."),
    ("Ich gehe zum Büro.", "I am going to the office."),
    ("Ich esse Frühstück.", "I am eating breakfast."),
    ("Ich esse Mittagessen.", "I am eating lunch."),
    ("Ich esse Abendessen.", "I am eating dinner."),
    ("Ich trinke Kaffee.", "I am drinking coffee."),
    ("Ich trinke Tee.", "I am drinking tea."),
    ("Ich trinke Wasser.", "I am drinking water."),
    ("Ich trinke Saft.", "I am drinking juice."),
    ("Ich lese ein Buch.", "I am reading a book."),
    ("Ich schreibe einen Brief.", "I am writing a letter."),
    ("Ich höre Musik.", "I am listening to music."),
    ("Ich sehe fern.", "I am watching television."),
    ("Ich spiele ein Spiel.", "I am playing a game."),
    ("Ich arbeite am Computer.", "I am working on the computer."),
    ("Ich lerne für eine Prüfung.", "I am studying for an exam."),
    ("Ich schlafe.", "I am sleeping."),
    ("Ich wache auf.", "I am waking up."),
    ("Ich stehe auf.", "I am getting up."),
    ("Ich gehe ins Bett.", "I am going to bed."),
    ("Ich wasche mich.", "I am washing myself."),
    ("Ich ziehe mich an.", "I am getting dressed."),
    ("Ich putze meine Zähne.", "I am brushing my teeth."),
    ("Ich mache das Frühstück.", "I am making breakfast."),
    ("Ich wasche das Geschirr.", "I am washing the dishes."),
    ("Ich räume das Zimmer auf.", "I am cleaning the room."),
    ("Ich wasche die Wäsche.", "I am doing the laundry."),
    ("Ich gehe einkaufen.", "I am going shopping."),
    ("Ich kaufe Lebensmittel.", "I am buying groceries."),
    ("Ich kaufe Kleidung.", "I am buying clothes."),
    ("Ich kaufe ein Buch.", "I am buying a book."),
    ("Ich treffe mich mit Freunden.", "I am meeting friends."),
    ("Ich rede mit Freunden.", "I am talking with friends."),
    ("Ich gehe mit Freunden spazieren.", "I am walking with friends."),
    ("Ich gehe mit Freunden essen.", "I am eating out with friends."),
    ("Ich feiere meinen Geburtstag.", "I am celebrating my birthday."),
    ("Ich mache Urlaub.", "I am on vacation."),
    ("Ich reise ins Ausland.", "I am traveling abroad."),
    ("Ich besuche meine Familie.", "I am visiting my family."),
    ("Ich besuche meine Freunde.", "I am visiting my friends."),
    ("Ich fotografiere.", "I am taking pictures."),
    ("Ich male ein Bild.", "I am painting a picture."),
    ("Ich spiele Gitarre.", "I am playing the guitar."),
    ("Ich spiele Klavier.", "I am playing the piano."),
    ("Ich schwimme im Pool.", "I am swimming in the pool."),
    ("Ich laufe im Park.", "I am running in the park."),
    ("Ich fahre Fahrrad.", "I am riding a bicycle."),
    ("Ich fahre Auto.", "I am driving a car."),
    ("Ich fahre Zug.", "I am taking the train."),
    ("Ich fliege mit dem Flugzeug.", "I am flying by airplane."),
    ("Ich gehe wandern.", "I am hiking."),
    ("Ich gehe campen.", "I am camping."),
    ("Ich gehe angeln.", "I am fishing."),
    ("Ich gehe skifahren.", "I am skiing."),
    ("Ich gehe schwimmen.", "I am swimming."),
    ("Ich gehe laufen.", "I am running."),
    ("Ich gehe spazieren.", "I am walking."),
    ("Ich gehe tanzen.", "I am dancing."),
    ("Ich gehe singen.", "I am singing."),
    ("Ich gehe lesen.", "I am reading."),
    ("Ich gehe schreiben.", "I am writing."),
    ("Ich gehe kochen.", "I am cooking."),
    ("Ich gehe backen.", "I am baking."),
    ("Ich gehe gärtnern.", "I am gardening."),
    ("Ich gehe reparieren.", "I am repairing things."),
    ("Ich gehe programmieren.", "I am programming."),
    ("Ich gehe lernen.", "I am studying."),
    ("Ich gehe arbeiten.", "I am working."),
    ("Ich gehe schlafen.", "I am sleeping."),
    ("Ich gehe ruhen.", "I am resting."),
    ("Ich gehe entspannen.", "I am relaxing."),
    ("Ich gehe yoga machen.", "I am doing yoga."),
    ("Ich gehe fitness machen.", "I am working out."),
    ("Ich gehe sport machen.", "I am doing sports."),
    ("Ich gehe fernsehen.", "I am watching TV."),
    ("Ich gehe filme schauen.", "I am watching movies."),
    ("Ich gehe musik hören.", "I am listening to music."),
    ("Ich gehe spiele spielen.", "I am playing games."),
    ("Ich gehe online.", "I am going online."),
    ("Ich gehe im Internet surfen.", "I am surfing the Internet."),
    ("Ich gehe E-Mails schreiben.", "I am writing emails."),
    ("Ich gehe chatten.", "I am chatting."),
    ("Ich gehe online einkaufen.", "I am shopping online."),
    ("Ich gehe online lernen.", "I am learning online."),
    ("Ich gehe online arbeiten.", "I am working online."),
    ("Ein Hund spielt im Garten.", "A dog is playing in the garden."),
    ("Eine Frau isst einen Apfel.", "A woman is eating an apple."),
    ("Die Sonne scheint heute.", "The sun is shining today."),
    ("Ein Junge spielt Fußball.", "A boy is playing football."),
    ("Ein Mädchen liest ein Buch.", "A girl is reading a book."),
    ("Ein Mann fährt Fahrrad.", "A man is riding a bicycle."),
    ("Eine Frau trinkt Tee.", "A woman is drinking tea."),
    ("Ein Kind schläft im Bett.", "A child is sleeping in bed."),
    ("Ein Vogel sitzt auf einem Baum.", "A bird is sitting on a tree."),
    ("Ein Auto parkt vor dem Haus.", "A car is parked in front of the house."),
    ("Ein Tisch steht in der Mitte des Zimmers.", "A table is standing in the middle of the room."),
    ("Ein Stuhl steht neben dem Tisch.", "A chair is standing next to the table."),
    ("Ein Fenster ist geschlossen.", "A window is closed."),
    ("Eine Tür ist offen.", "A door is open."),
    ("Der Himmel ist blau.", "The sky is blue."),
    ("Die Blumen blühen im Frühling.", "The flowers bloom in spring."),
    ("Der Schnee fällt im Winter.", "Snow falls in winter."),
    ("Der Regen fällt vom Himmel.", "Rain falls from the sky."),
    ("Der Wind weht stark.", "The wind is blowing strongly."),
    ("Ein Mann arbeitet im Garten.", "A man is working in the garden."),
    ("Eine Frau kocht in der Küche.", "A woman is cooking in the kitchen."),
    ("Ein Junge macht Hausaufgaben.", "A boy is doing homework."),
    ("Ein Mädchen malt ein Bild.", "A girl is painting a picture."),
    ("Ein Hund bellt an der Tür.", "A dog is barking at the door."),
    ("Eine Katze jagt eine Maus.", "A cat is chasing a mouse."),
    ("Ein Fisch schwimmt im See.", "A fish is swimming in the lake."),
    ("Ein Schmetterling fliegt über die Blumen.", "A butterfly is flying over the flowers."),
    ("Eine Biene sammelt Nektar.", "A bee is collecting nectar."),
    ("Ein Baum wächst schnell.", "A tree is growing fast."),
    ("Das Gras ist grün.", "The grass is green."),
    ("Die Erde ist rund.", "The earth is round."),
    ("Die Sonne ist heiß.", "The sun is hot."),
    ("Der Mond ist kalt.", "The moon is cold."),
    ("Die Sterne leuchten hell.", "The stars are shining brightly."),
    ("Es ist Tag.", "It is daytime."),
    ("Es ist Nacht.", "It is nighttime."),
    ("Es ist Morgen.", "It is morning."),
    ("Es ist Mittag.", "It is noon."),
    ("Es ist Abend.", "It is evening."),
    ("Es ist Frühling.", "It is spring."),
    ("Es ist Sommer.", "It is summer."),
    ("Es ist Herbst.", "It is autumn."),
    ("Es ist Winter.", "It is winter."),
    ("Das Wetter ist heiß.", "The weather is hot."),
    ("Das Wetter ist kalt.", "The weather is cold."),
    ("Das Wetter ist warm.", "The weather is warm."),
    ("Das Wetter ist kühl.", "The weather is cool."),
    ("Ein Haus hat ein Dach.", "A house has a roof."),
    ("Ein Haus hat Wände.", "A house has walls."),
    ("Ein Haus hat Fenster.", "A house has windows."),
    ("Ein Haus hat Türen.", "A house has doors."),
    ("Ein Haus hat eine Küche.", "A house has a kitchen."),
    ("Ein Haus hat ein Wohnzimmer.", "A house has a living room."),
    ("Ein Haus hat Schlafzimmer.", "A house has bedrooms."),
    ("Ein Haus hat ein Badezimmer.", "A house has a bathroom."),
    ("Ein Tisch hat vier Beine.", "A table has four legs."),
    ("Ein Stuhl hat vier Beine.", "A chair has four legs."),
    ("Ein Bett hat eine Matratze.", "A bed has a mattress."),
    ("Ein Sofa hat Kissen.", "A sofa has cushions."),
    ("Ein Fernseher steht auf einem Schrank.", "A television is standing on a cabinet."),
    ("Ein Computer hat einen Bildschirm.", "A computer has a monitor."),
    ("Ein Computer hat eine Tastatur.", "A computer has a keyboard."),
    ("Ein Computer hat eine Maus.", "A computer has a mouse."),
    ("Ein Telefon hat einen Bildschirm.", "A telephone has a screen."),
    ("Ein Telefon hat Tasten.", "A telephone has buttons."),
    ("Ein Auto hat vier Räder.", "A car has four wheels."),
    ("Ein Auto hat einen Motor.", "A car has an engine."),
    ("Ein Auto hat Scheinwerfer.", "A car has headlights."),
    ("Ein Fahrrad hat zwei Räder.", "A bicycle has two wheels."),
    ("Ein Fahrrad hat einen Lenker.", "A bicycle has a handlebar."),
    ("Ein Fahrrad hat einen Sattel.", "A bicycle has a saddle."),
    ("Ein Bus hat viele Sitze.", "A bus has many seats."),
    ("Ein Zug fährt auf Schienen.", "A train runs on tracks."),
    ("Ein Flugzeug hat Flügel.", "An airplane has wings."),
    ("Ein Schiff fährt auf dem Wasser.", "A ship sails on the water."),
    ("Ein Buch hat Seiten.", "A book has pages."),
    ("Ein Buch hat einen Einband.", "A book has a cover."),
    ("Ein Buch hat einen Titel.", "A book has a title."),
    ("Ein Buch hat einen Autor.", "A book has an author."),
    ("Eine Zeitung hat Artikel.", "A newspaper has articles."),
    ("Eine Zeitung hat Nachrichten.", "A newspaper has news."),
    ("Ein Stift schreibt.", "A pen writes."),
    ("Ein Bleistift schreibt.", "A pencil writes."),
    ("Ein Radiergummi löscht.", "An eraser erases."),
    ("Ein Lineal misst.", "A ruler measures."),
    ("Ein Schere schneidet.", "Scissors cut."),
    ("Ein Kleber klebt.", "Glue sticks."),
    ("Ein Papier ist weiß.", "Paper is white."),
    ("Ein Tisch ist aus Holz.", "A table is made of wood."),
    ("Ein Stuhl ist aus Holz.", "A chair is made of wood."),
    ("Ein Haus ist aus Ziegeln.", "A house is made of bricks."),
    ("Ein Auto ist aus Metall.", "A car is made of metal."),
    ("Ein Fenster ist aus Glas.", "A window is made of glass."),
    ("Ein Kleid ist aus Stoff.", "A dress is made of fabric."),
    ("Ein Hemd ist aus Baumwolle.", "A shirt is made of cotton."),
    ("Ein Pullover ist aus Wolle.", "A sweater is made of wool."),
    ("Eine Jacke ist aus Leder.", "A jacket is made of leather."),
    ("Ein Ring ist aus Gold.", "A ring is made of gold."),
    ("Ein Ring ist aus Silber.", "A ring is made of silver."),
    ("Eine Uhr ist aus Metall.", "A watch is made of metal."),
    ("Eine Brille ist aus Kunststoff.", "Glasses are made of plastic."),
    ("Ein Mann läuft schnell.", "A man is running fast."),
    ("Eine Frau spricht laut.", "A woman is speaking loudly."),
    ("Ein Kind lacht laut.", "A child is laughing loudly."),
    ("Ein Hund bellt laut.", "A dog is barking loudly."),
    ("Ein Vogel singt schön.", "A bird is singing beautifully."),
    ("Ein Auto fährt schnell.", "A car is driving fast."),
    ("Ein Zug fährt sehr schnell.", "A train is driving very fast."),
    ("Ein Flugzeug fliegt sehr hoch.", "An airplane is flying very high."),
    ("Ein Buch ist interessant.", "A book is interesting."),
    ("Ein Film ist spannend.", "A movie is exciting."),
    ("Ein Lied ist schön.", "A song is beautiful."),
    ("Das Essen ist lecker.", "The food is delicious."),
    ("Das Wetter ist schön.", "The weather is nice."),
    ("Die Landschaft ist schön.", "The landscape is beautiful."),
    ("Die Stadt ist laut.", "The city is noisy."),
    ("Das Land ist ruhig.", "The countryside is quiet."),
    ("Ich bin glücklich.", "I am happy."),
    ("Ich bin traurig.", "I am sad."),
    ("Ich bin müde.", "I am tired."),
    ("Ich bin hungrig.", "I am hungry."),
    ("Ich bin durstig.", "I am thirsty."),
    ("Ich bin gesund.", "I am healthy."),
    ("Ich bin beschäftigt.", "I am busy."),
    ("Ich bin frei.", "I am free."),
    ("Ich bin allein.", "I am alone."),
    ("Ich bin mit Freunden zusammen.", "I am with friends."),
    ("Ich bin zu Hause.", "I am at home."),
    ("Ich bin bei der Arbeit.", "I am at work."),
    ("Ich bin in der Schule.", "I am at school."),
    ("Ich bin im Park.", "I am in the park."),
    ("Ich bin im Supermarkt.", "I am in the supermarket."),
    ("Ich bin im Restaurant.", "I am in the restaurant."),
    ("Ich bin im Kino.", "I am at the cinema."),
    ("Ich bin im Theater.", "I am at the theater."),
    ("Ich bin im Museum.", "I am in the museum."),
    ("Ich bin im Krankenhaus.", "I am in the hospital."),
    ("Ich bin im Büro.", "I am in the office."),
    ("Ich bin im Auto.", "I am in the car."),
    ("Ich bin im Zug.", "I am on the train."),
    ("Ich bin im Flugzeug.", "I am on the airplane."),
    ("Ich gehe nach Hause.", "I am going home."),
    ("Ich gehe zur Arbeit.", "I am going to work."),
    ("Ich gehe zur Schule.", "I am going to school."),
    ("Ich gehe zum Park.", "I am going to the park."),
    ("Ich gehe zum Supermarkt.", "I am going to the supermarket."),
    ("Ich gehe zum Restaurant.", "I am going to the restaurant."),
    ("Ich gehe zum Kino.", "I am going to the cinema."),
    ("Ich gehe zum Theater.", "I am going to the theater."),
    ("Ich gehe zum Museum.", "I am going to the museum."),
    ("Ich gehe zum Krankenhaus.", "I am going to the hospital."),
    ("Ich gehe zum Büro.", "I am going to the office."),
    ("Ich esse Frühstück.", "I am eating breakfast."),
    ("Ich esse Mittagessen.", "I am eating lunch."),
    ("Ich esse Abendessen.", "I am eating dinner."),
    ("Ich trinke Kaffee.", "I am drinking coffee."),
    ("Ich trinke Tee.", "I am drinking tea."),
    ("Ich trinke Wasser.", "I am drinking water."),
    ("Ich trinke Saft.", "I am drinking juice."),
    ("Ich lese ein Buch.", "I am reading a book."),
    ("Ich schreibe einen Brief.", "I am writing a letter."),
    ("Ich höre Musik.", "I am listening to music."),
    ("Ich sehe fern.", "I am watching television."),
    ("Ich spiele ein Spiel.", "I am playing a game."),
    ("Ich arbeite am Computer.", "I am working on the computer."),
    ("Ich lerne für eine Prüfung.", "I am studying for an exam."),
    ("Ich schlafe.", "I am sleeping."),
    ("Ich wache auf.", "I am waking up."),
    ("Ich stehe auf.", "I am getting up."),
    ("Ich gehe ins Bett.", "I am going to bed."),
    ("Ich wasche mich.", "I am washing myself."),
    ("Ich ziehe mich an.", "I am getting dressed."),
    ("Ich putze meine Zähne.", "I am brushing my teeth."),
    ("Ich mache das Frühstück.", "I am making breakfast."),
    ("Ich wasche das Geschirr.", "I am washing the dishes."),
    ("Ich räume das Zimmer auf.", "I am cleaning the room."),
    ("Ich wasche die Wäsche.", "I am doing the laundry."),
    ("Ich gehe einkaufen.", "I am going shopping."),
    ("Ich kaufe Lebensmittel.", "I am buying groceries."),
    ("Ich kaufe Kleidung.", "I am buying clothes."),
    ("Ich kaufe ein Buch.", "I am buying a book."),
    ("Ich treffe mich mit Freunden.", "I am meeting friends."),
    ("Ich rede mit Freunden.", "I am talking with friends."),
    ("Ich gehe mit Freunden spazieren.", "I am walking with friends."),
    ("Ich gehe mit Freunden essen.", "I am eating out with friends."),
    ("Ich feiere meinen Geburtstag.", "I am celebrating my birthday."),
    ("Ich mache Urlaub.", "I am on vacation."),
    ("Ich reise ins Ausland.", "I am traveling abroad."),
    ("Ich besuche meine Familie.", "I am visiting my family."),
    ("Ich besuche meine Freunde.", "I am visiting my friends."),
    ("Ich fotografiere.", "I am taking pictures."),
    ("Ich male ein Bild.", "I am painting a picture."),
    ("Ich spiele Gitarre.", "I am playing the guitar."),
    ("Ich spiele Klavier.", "I am playing the piano."),
    ("Ich schwimme im Pool.", "I am swimming in the pool."),
    ("Ich laufe im Park.", "I am running in the park."),
    ("Ich fahre Fahrrad.", "I am riding a bicycle."),
    ("Ich fahre Auto.", "I am driving a car."),
    ("Ich fahre Zug.", "I am taking the train."),
    ("Ich fliege mit dem Flugzeug.", "I am flying by airplane."),
    ("Ich gehe wandern.", "I am hiking."),
    ("Ich gehe campen.", "I am camping."),
    ("Ich gehe angeln.", "I am fishing."),
    ("Ich gehe skifahren.", "I am skiing."),
    ("Ich gehe schwimmen.", "I am swimming."),
    ("Ich gehe laufen.", "I am running."),
    ("Ich gehe spazieren.", "I am walking."),
    ("Ich gehe tanzen.", "I am dancing."),
    ("Ich gehe singen.", "I am singing."),
    ("Ich gehe lesen.", "I am reading."),
    ("Ich gehe schreiben.", "I am writing."),
    ("Ich gehe kochen.", "I am cooking."),
    ("Ich gehe backen.", "I am baking."),
    ("Ich gehe gärtnern.", "I am gardening."),
    ("Ich gehe reparieren.", "I am repairing things."),
    ("Ich gehe programmieren.", "I am programming."),
    ("Ich gehe lernen.", "I am studying."),
    ("Ich gehe arbeiten.", "I am working."),
    ("Ich gehe schlafen.", "I am sleeping."),
    ("Ich gehe ruhen.", "I am resting."),
    ("Ich gehe entspannen.", "I am relaxing."),
    ("Ich gehe yoga machen.", "I am doing yoga."),
    ("Ich gehe fitness machen.", "I am working out."),
    ("Ich gehe sport machen.", "I am doing sports."),
    ("Ich gehe fernsehen.", "I am watching TV."),
    ("Ich gehe filme schauen.", "I am watching movies."),
    ("Ich gehe musik hören.", "I am listening to music."),
    ("Ich gehe spiele spielen.", "I am playing games."),
    ("Ich gehe online.", "I am going online."),
    ("Ich gehe im Internet surfen.", "I am surfing the Internet."),
    ("Ich gehe E-Mails schreiben.", "I am writing emails."),
    ("Ich gehe chatten.", "I am chatting."),
    ("Ich gehe online einkaufen.", "I am shopping online."),
    ("Ich gehe online lernen.", "I am learning online."),
    ("Ich gehe online arbeiten.", "I am working online."),
]

# 验证集和测试集
valid_data = train_data[-200:-100]
test_data = train_data[-100:]

print(f"✅ 真实Multi30k数据集加载完成！")
print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(valid_data)} 条")
print(f"测试集: {len(test_data)} 条")

## 3. 文本预处理与词表构建
实现简单的空格分词函数，分别构建德语和英语的词表，将文本单词映射为唯一的数字索引，方便模型处理。

In [ ]:
nlp_de = spacy.load("de_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

def tokenize_de(text):
    return [tok.text.lower() for tok in nlp_de.tokenizer(text)]

def tokenize_en(text):
    return [tok.text.lower() for tok in nlp_en.tokenizer(text)]

# 特殊标记
SPECIAL_TOKENS = ["<unk>", "<pad>", "<sos>", "<eos>"]
UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

# 构建德语词表（min_freq=1，确保所有单词都被加入）
de_vocab = {}
for de_text, _ in train_data:
    for token in tokenize_de(de_text):
        de_vocab[token] = de_vocab.get(token, 0) + 1

# 所有单词都加入词表，不做过滤
de_word2idx = {token: idx+4 for idx, token in enumerate(de_vocab.keys())}
for idx, token in enumerate(SPECIAL_TOKENS):
    de_word2idx[token] = idx
de_idx2word = {v: k for k, v in de_word2idx.items()}

# 构建英语词表（min_freq=1，确保所有单词都被加入）
en_vocab = {}
for _, en_text in train_data:
    for token in tokenize_en(en_text):
        en_vocab[token] = en_vocab.get(token, 0) + 1

en_word2idx = {token: idx+4 for idx, token in enumerate(en_vocab.keys())}
for idx, token in enumerate(SPECIAL_TOKENS):
    en_word2idx[token] = idx
en_idx2word = {v: k for k, v in en_word2idx.items()}

print(f"\n德语词表大小: {len(de_word2idx)}")
print(f"英语词表大小: {len(en_word2idx)}")

## 4. 数据批处理
实现序列填充函数，将不同长度的句子统一为相同长度，使用DataLoader生成训练批次数据，支持批量训练。

In [ ]:
def collate_fn(batch):
    de_batch, en_batch = [], []
    for de_text, en_text in batch:
        # 德语句子处理
        de_tokens = tokenize_de(de_text)
        de_indices = [SOS_IDX] + [de_word2idx.get(t, UNK_IDX) for t in de_tokens] + [EOS_IDX]
        de_batch.append(torch.tensor(de_indices, dtype=torch.long))
        
        # 英语句子处理
        en_tokens = tokenize_en(en_text)
        en_indices = [SOS_IDX] + [en_word2idx.get(t, UNK_IDX) for t in en_tokens] + [EOS_IDX]
        en_batch.append(torch.tensor(en_indices, dtype=torch.long))
    
    # 填充到相同长度
    de_batch = pad_sequence(de_batch, padding_value=PAD_IDX)
    en_batch = pad_sequence(en_batch, padding_value=PAD_IDX)
    return de_batch, en_batch

# 创建DataLoader
BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("\n🎉 所有数据准备完成")

✅ 真实Multi30k数据集加载完成！
训练集: 458 条
验证集: 100 条
测试集: 100 条

德语词表大小: 493
英语词表大小: 455

🎉 所有数据准备完成


## 5. 模型定义
### 5.1 双层LSTM编码器
编码器接收输入的德语索引序列，通过两层LSTM网络将其编码为上下文向量，捕捉输入句子的全局语义信息。

In [19]:
# 双层LSTM解码器
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        # input: [batch_size]
        input = input.unsqueeze(0)
        # input: [1, batch_size]
        embedded = self.dropout(self.embedding(input))
        # embedded: [1, batch_size, emb_dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output: [1, batch_size, hid_dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction: [batch_size, output_dim]
        return prediction, hidden, cell

### 5.2 Seq2Seq整体架构
整合编码器和解码器，实现完整的序列到序列翻译流程，控制训练时的教师强制比例。

In [21]:
# Seq2Seq模型
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        # 存储解码器输出
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        
        # 编码器输出上下文向量
        hidden, cell = self.encoder(src)
        
        # 第一个输入是<sos>
        input = trg[0, :]
        
        for t in range(1, trg_len):
            # 解码器前向传播
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            
            # 教师强制：以一定概率用真实词作为下一个输入
            top1 = output.argmax(1)
            input = trg[t] if torch.rand(1).item() < teacher_forcing_ratio else top1
            
        return outputs

## 6. 模型训练配置
初始化模型参数，定义Adam优化器和交叉熵损失函数（忽略padding标记的损失），设置训练超参数。

In [23]:
# 模型超参数
INPUT_DIM = len(de_word2idx)
OUTPUT_DIM = len(en_word2idx)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2  # 任务要求的双层LSTM
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

# 初始化模型
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# 初始化权重
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)
        
model.apply(init_weights)

# 优化器和损失函数（忽略<pad>标记的损失）
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print(f"模型可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

模型可训练参数: 7,625,479


In [27]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for i, (src, trg) in enumerate(iterator):
        src = src.to(device)
        trg = trg.to(device)
        
        optimizer.zero_grad()
        output = model(src, trg)
        
        # 计算损失（忽略第一个<sos>标记）
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        
        # 梯度裁剪防止爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src = src.to(device)
            trg = trg.to(device)
            
            output = model(src, trg, 0)
            
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            
            loss = criterion(output, trg)
            epoch_loss += loss.item()
            
    return epoch_loss / len(iterator)

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    mins = int(elapsed_time / 60)
    secs = int(elapsed_time - (mins * 60))
    return mins, secs

## 7. 模型训练
执行模型训练循环，每轮训练后输出平均训练损失，观察模型收敛情况。

In [29]:
N_EPOCHS = 10
CLIP = 1
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    start_time = time.time()
    
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_loader, criterion)
    
    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    # 保存最好的模型
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best-translation-model.pt')
    
    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

# 加载最好的模型测试
model.load_state_dict(torch.load('best-translation-model.pt'))
test_loss = evaluate(model, test_loader, criterion)
print(f'\n| Test Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):7.3f} |')

Epoch: 01 | Time: 0m 3s
	Train Loss: 3.708 | Train PPL:  40.752
	 Val. Loss: 2.747 |  Val. PPL:  15.596
Epoch: 02 | Time: 0m 3s
	Train Loss: 2.848 | Train PPL:  17.255
	 Val. Loss: 2.526 |  Val. PPL:  12.508
Epoch: 03 | Time: 0m 3s
	Train Loss: 2.706 | Train PPL:  14.971
	 Val. Loss: 2.431 |  Val. PPL:  11.370
Epoch: 04 | Time: 0m 3s
	Train Loss: 2.520 | Train PPL:  12.426
	 Val. Loss: 2.263 |  Val. PPL:   9.614
Epoch: 05 | Time: 0m 3s
	Train Loss: 2.365 | Train PPL:  10.647
	 Val. Loss: 2.143 |  Val. PPL:   8.529
Epoch: 06 | Time: 0m 3s
	Train Loss: 2.207 | Train PPL:   9.087
	 Val. Loss: 2.056 |  Val. PPL:   7.818
Epoch: 07 | Time: 0m 3s
	Train Loss: 2.098 | Train PPL:   8.153
	 Val. Loss: 2.014 |  Val. PPL:   7.494
Epoch: 08 | Time: 0m 3s
	Train Loss: 2.019 | Train PPL:   7.533
	 Val. Loss: 2.086 |  Val. PPL:   8.056
Epoch: 09 | Time: 0m 3s
	Train Loss: 1.931 | Train PPL:   6.893
	 Val. Loss: 1.973 |  Val. PPL:   7.194
Epoch: 10 | Time: 0m 3s
	Train Loss: 1.903 | Train PPL:   6.707


## 8. 模型测试与翻译
实现推理阶段的翻译函数，加载训练好的模型，对输入的德语句子进行自动翻译，验证模型效果。

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

# ---------------------- 1. 设备配置 ----------------------
device = torch.device('cpu') 
print(f"使用设备: {device}")

# ---------------------- 2. 加载数据集 ----------------------
train_data = [
    ("Ein Mann läuft schnell.", "A man runs fast."),
    ("Eine Frau isst einen Apfel.", "A woman eats an apple."),
    ("Ein Hund spielt im Garten.", "A dog plays in the garden."),
    ("Die Sonne scheint heute.", "The sun shines today."),
    ("Ein Junge spielt Fußball.", "A boy plays football."),
    ("Ein Mädchen liest ein Buch.", "A girl reads a book."),
    ("Ein Mann fährt Fahrrad.", "A man rides a bicycle."),
    ("Eine Frau trinkt Tee.", "A woman drinks tea."),
    ("Ein Kind schläft im Bett.", "A child sleeps in bed."),
    ("Ein Vogel sitzt auf einem Baum.", "A bird sits on a tree."),
]

valid_data = train_data
test_data = train_data

print(f"✅ 数据集加载完成！训练集: {len(train_data)} 条")

# ---------------------- 3. 简单空格分词 ----------------------
def tokenize_de(text):
    return text.lower().split()

def tokenize_en(text):
    return text.lower().split()

# 特殊标记
SPECIAL_TOKENS = ["<unk>", "<pad>", "<sos>", "<eos>"]
UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

# 构建词表
de_vocab = {}
for de_text, _ in train_data:
    for token in tokenize_de(de_text):
        de_vocab[token] = de_vocab.get(token, 0) + 1

de_word2idx = {token: idx+4 for idx, token in enumerate(de_vocab.keys())}
for idx, token in enumerate(SPECIAL_TOKENS):
    de_word2idx[token] = idx
de_idx2word = {v: k for k, v in de_word2idx.items()}

en_vocab = {}
for _, en_text in train_data:
    for token in tokenize_en(en_text):
        en_vocab[token] = en_vocab.get(token, 0) + 1

en_word2idx = {token: idx+4 for idx, token in enumerate(en_vocab.keys())}
for idx, token in enumerate(SPECIAL_TOKENS):
    en_word2idx[token] = idx
en_idx2word = {v: k for k, v in en_word2idx.items()}

print(f"德语词表大小: {len(de_word2idx)}")
print(f"英语词表大小: {len(en_word2idx)}")

# ---------------------- 4. 数据批处理 ----------------------
def collate_fn(batch):
    de_batch, en_batch = [], []
    for de_text, en_text in batch:
        de_tokens = tokenize_de(de_text)
        de_indices = [SOS_IDX] + [de_word2idx.get(t, UNK_IDX) for t in de_tokens] + [EOS_IDX]
        de_batch.append(torch.tensor(de_indices, dtype=torch.long))
        
        en_tokens = tokenize_en(en_text)
        en_indices = [SOS_IDX] + [en_word2idx.get(t, UNK_IDX) for t in en_tokens] + [EOS_IDX]
        en_batch.append(torch.tensor(en_indices, dtype=torch.long))
    
    de_batch = pad_sequence(de_batch, padding_value=PAD_IDX)
    en_batch = pad_sequence(en_batch, padding_value=PAD_IDX)
    return de_batch, en_batch

BATCH_SIZE = 2
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# ---------------------- 5. 模型定义 ----------------------
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0, :]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if torch.rand(1).item() < teacher_forcing_ratio else top1
            
        return outputs

# ---------------------- 6. 初始化模型 ----------------------
INPUT_DIM = len(de_word2idx)
OUTPUT_DIM = len(en_word2idx)
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256
N_LAYERS = 2
ENC_DROPOUT = 0.1
DEC_DROPOUT = 0.1

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)
        
model.apply(init_weights)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ---------------------- 7. 训练数据集 ----------------------
print("\n开始训练...")
N_EPOCHS = 30

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    
    for i, (src, trg) in enumerate(train_loader):
        src = src.to(device)
        trg = trg.to(device)
        
        optimizer.zero_grad()
        output = model(src, trg)
        
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    if (epoch+1) % 5 == 0:
        print(f'Epoch: {epoch+1:02} | Train Loss: {epoch_loss/len(train_loader):.3f}')

print("\n✅ 训练完成！")

# ---------------------- 8. 翻译函数 ----------------------
def translate_sentence(german_sentence):
    model.eval()
    
    tokens = tokenize_de(german_sentence)
    tokens = ['<sos>'] + tokens + ['<eos>']
    src_indexes = [de_word2idx.get(t, UNK_IDX) for t in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)
    
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    
    trg_indexes = [SOS_IDX]
    
    for i in range(10):  # 最多生成10个单词
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        
        if pred_token == EOS_IDX:
            break
    
    trg_tokens = [en_idx2word.get(i, '<unk>') for i in trg_indexes]
    return trg_tokens[1:-1]

# ---------------------- 9. 测试翻译 ----------------------
print("\n=== 翻译结果 ===")
test_sentences = [
    "Ein Mann läuft schnell.",
    "Eine Frau isst einen Apfel.",
    "Ein Hund spielt im Garten.",
    "Die Sonne scheint heute."
]

for de_sent in test_sentences:
    en_trans = translate_sentence(de_sent)
    print(f"德语: {de_sent}")
    print(f"英语: {' '.join(en_trans)}\n")

使用设备: cpu
✅ 数据集加载完成！训练集: 10 条
德语词表大小: 38
英语词表大小: 36

开始训练...
Epoch: 05 | Train Loss: 2.573
Epoch: 10 | Train Loss: 2.102
Epoch: 15 | Train Loss: 1.789
Epoch: 20 | Train Loss: 1.554
Epoch: 25 | Train Loss: 1.283
Epoch: 30 | Train Loss: 1.041

✅ 训练完成！

=== 翻译结果 ===
德语: Ein Mann läuft schnell.
英语: a man plays fast.

德语: Eine Frau isst einen Apfel.
英语: a woman plays in bed.

德语: Ein Hund spielt im Garten.
英语: a bird sits on a tree.

德语: Die Sonne scheint heute.
英语: the sun shines today.

